# Notebook 11: Advanced Optimizers — SGD, Momentum, Adagrad, RMSProp & Adam

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 3 hours  
**Theme:** House Price Classification (above / below median) — California style  
**Week 8 — Regularization, Hyperparameter Tuning & Optimization**

---

## Real-World Analogy: Navigating a Hilly Landscape in Fog

Imagine you are dropped onto a hilly terrain at night — dense fog, no map, only a flashlight that shows the slope right beneath your feet.  
Your goal: reach the lowest valley (minimum loss).

| Optimizer | Analogy | Key Behavior |
|-----------|---------|-------------|
| **SGD** | You take one step in whichever direction feels downhill *right now* | Gets stuck in narrow ravines; oscillates on steep slopes |
| **Momentum** | Like a heavy ball rolling downhill — it builds speed | Rolls through shallow bumps; overshoots if too fast |
| **Adagrad** | Each foot takes steps scaled to its own terrain history | Rare-feature parameters take big steps; frequent ones take small steps — eventually freezes |
| **RMSProp** | Like Adagrad but with amnesia — forgets distant history | Maintains useful learning rates throughout training |
| **Adam** | Combines the momentum ball + per-parameter adaptive steps | Fast, stable, most widely used in practice |

---

## Learning Objectives
1. Understand *why* SGD fails in pathological loss landscapes  
2. Derive and implement Momentum, Adagrad, RMSProp, and Adam from scratch  
3. Visualize convergence curves and 2-D loss landscape trajectories  
4. Analyse learning-rate sensitivity and hyperparameter robustness  
5. Understand bias correction in Adam and its importance in early iterations

In [ ]:
# ── Cell 2: Imports & reproducibility ─────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Reproducibility — always set before any random calls
np.random.seed(42)

# Plot aesthetics
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# Optimizer colour map used throughout the notebook
OPT_COLORS = {
    "SGD":      "#e41a1c",
    "Momentum": "#ff7f00",
    "Adagrad":  "#4daf4a",
    "RMSProp":  "#984ea3",
    "Adam":     "#377eb8",
}

print("Imports successful.  NumPy:", np.__version__)

## Section 1 — Dataset: Binary Classification from House Prices

We use the **California Housing** dataset and convert it into a binary classification problem:  
**Target = 1** if the house price is above the dataset median, **Target = 0** otherwise.

This gives a logistic regression task with a clean sigmoid loss function — ideal for comparing optimizers.

In [ ]:
# ── Cell 4: Load & prepare data ────────────────────────────────────────────────
# Fetch the raw California housing data
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Binary target: 1 = above median price, 0 = below median price
median_price = df["MedHouseVal"].median()
df["target"] = (df["MedHouseVal"] > median_price).astype(int)

print(f"Dataset shape     : {df.shape}")
print(f"Median house value: ${median_price:.3f} (×100k)")
print(f"Class balance     : {df['target'].mean()*100:.1f}% above median")

# Features and labels
feature_cols = housing.feature_names
X_raw = df[feature_cols].values
y     = df["target"].values

# Train/test split before scaling (no data leakage)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise features: mean=0, std=1 (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

# Add bias column (column of 1s) so the intercept is part of theta
X_train_b = np.c_[np.ones(len(X_train)), X_train]   # shape (N, p+1)
X_test_b  = np.c_[np.ones(len(X_test)),  X_test]

print(f"\nTraining set shape: {X_train_b.shape}")
print(f"Test set shape    : {X_test_b.shape}")

## Section 2 — Implementing All Five Optimizers from Scratch

### 2a. Shared Helper: Logistic Loss & Gradient

All optimizers minimise the **binary cross-entropy** (logistic loss):

$$\mathcal{L}(\theta) = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\right]$$

Gradient: $\nabla_\theta \mathcal{L} = \frac{1}{N} X^\top (\hat{y} - y)$

The sigmoid is clipped to prevent overflow in `exp`.

In [ ]:
# ── Cell 6: Shared helper functions ───────────────────────────────────────────
def sigmoid(z):
    """Numerically stable sigmoid: clip z to [-500, 500] before exp."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def logistic_loss(X, y, theta):
    """
    Binary cross-entropy loss.
    Adds 1e-10 inside log to avoid log(0) when predictions are exactly 0 or 1.
    """
    y_hat = sigmoid(X @ theta)
    return -np.mean(
        y * np.log(y_hat + 1e-10) + (1 - y) * np.log(1 - y_hat + 1e-10)
    )

def logistic_gradient(X, y, theta):
    """
    Full-batch gradient of the binary cross-entropy loss.
    ∇L = (1/N) * X^T @ (y_hat - y)
    """
    n = X.shape[0]
    y_hat = sigmoid(X @ theta)
    return (1.0 / n) * (X.T @ (y_hat - y))

# Quick sanity check with random theta
theta_test = np.zeros(X_train_b.shape[1])
loss0 = logistic_loss(X_train_b, y_train, theta_test)
grad0 = logistic_gradient(X_train_b, y_train, theta_test)

print(f"Initial loss (theta=0): {loss0:.4f}  — expected ≈ ln(2) ≈ 0.693")
print(f"Gradient norm         : {np.linalg.norm(grad0):.4f}")

### 2b. SGD (Vanilla Gradient Descent)

The simplest rule: move a fixed step in the negative gradient direction.

$$\theta \leftarrow \theta - \alpha \nabla \mathcal{L}(\theta)$$

**Problems:**
- Same step size for every parameter, every iteration  
- Oscillates in ravines (where curvature differs across axes)  
- Sensitive to the choice of learning rate $\alpha$

In [ ]:
# ── Cell 8: SGD optimizer ──────────────────────────────────────────────────────
def sgd_optimizer(X, y, lr=0.1, n_epochs=200):
    """
    Vanilla (full-batch) Stochastic Gradient Descent.

    Update rule:  theta  ←  theta - lr * ∇L(theta)

    Parameters
    ----------
    X        : feature matrix (N, p)
    y        : binary labels (N,)
    lr       : learning rate α
    n_epochs : number of gradient steps

    Returns
    -------
    theta  : final parameter vector
    losses : loss recorded after each epoch
    """
    n, p = X.shape
    theta  = np.zeros(p)    # initialise at zero
    losses = []

    for epoch in range(n_epochs):
        grad   = logistic_gradient(X, y, theta)
        theta -= lr * grad                        # vanilla step
        losses.append(logistic_loss(X, y, theta))

    return theta, losses

# Run and report
theta_sgd, losses_sgd = sgd_optimizer(X_train_b, y_train, lr=0.1, n_epochs=200)
acc_sgd = accuracy_score(y_test, (sigmoid(X_test_b @ theta_sgd) >= 0.5).astype(int))
print(f"SGD  — Final loss: {losses_sgd[-1]:.4f}  |  Test accuracy: {acc_sgd*100:.2f}%")

### 2c. SGD with Momentum

Instead of taking a fresh step every iteration, we accumulate a *velocity* $v$ and let it carry us through noisy gradients.

$$v \leftarrow \beta v - \alpha \nabla \mathcal{L} \qquad \theta \leftarrow \theta + v$$

With $\beta = 0.9$: 90 % of last velocity is preserved — the ball remembers direction.

In [ ]:
# ── Cell 10: Momentum optimizer ───────────────────────────────────────────────
def momentum_optimizer(X, y, lr=0.1, beta=0.9, n_epochs=200):
    """
    SGD with Momentum.

    Velocity update:   v  ←  β·v  -  lr·∇L
    Parameter update:  θ  ←  θ  +  v

    β = 0.9 means "remember 90% of previous velocity".
    Effect: accelerates in consistent directions, dampens oscillation in ravines.

    Parameters
    ----------
    beta : momentum coefficient (typically 0.9)
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    v      = np.zeros(p)   # velocity vector, same shape as theta
    losses = []

    for epoch in range(n_epochs):
        grad  = logistic_gradient(X, y, theta)

        # Velocity accumulates gradient history — like a rolling ball
        v     = beta * v - lr * grad
        theta = theta + v

        losses.append(logistic_loss(X, y, theta))

    return theta, losses

theta_mom, losses_mom = momentum_optimizer(X_train_b, y_train, lr=0.1, beta=0.9)
acc_mom = accuracy_score(y_test, (sigmoid(X_test_b @ theta_mom) >= 0.5).astype(int))
print(f"Momentum — Final loss: {losses_mom[-1]:.4f}  |  Test accuracy: {acc_mom*100:.2f}%")

### 2d. Adagrad (Adaptive Gradient)

Different parameters should learn at different rates — frequent features already have large gradients; rare ones need bigger steps.

$$G_j \mathrel{+}= (\nabla \mathcal{L}_j)^2 \qquad \theta_j \leftarrow \theta_j - \frac{\alpha}{\sqrt{G_j + \varepsilon}} \nabla \mathcal{L}_j$$

**Problem:** $G_j$ grows monotonically → effective learning rate $\to 0$ → training stops prematurely.

In [ ]:
# ── Cell 12: Adagrad optimizer ────────────────────────────────────────────────
def adagrad_optimizer(X, y, lr=0.5, eps=1e-8, n_epochs=200):
    """
    Adagrad — Adaptive Gradient Algorithm (Duchi et al., 2011).

    Accumulates squared gradients in G (one value per parameter).
    Effective learning rate: α / sqrt(G_j + ε).

    Upside : no manual per-feature tuning needed.
    Downside: G never decreases → lr eventually collapses to ~0.
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    G      = np.zeros(p)   # accumulated squared gradients
    losses = []

    for epoch in range(n_epochs):
        grad  = logistic_gradient(X, y, theta)

        # Accumulate squared gradient — G grows monotonically
        G    += grad ** 2

        # Per-parameter update: parameters with large past gradients get small steps
        theta -= (lr / (np.sqrt(G) + eps)) * grad

        losses.append(logistic_loss(X, y, theta))

    return theta, losses

theta_ada, losses_ada = adagrad_optimizer(X_train_b, y_train, lr=0.5, n_epochs=200)
acc_ada = accuracy_score(y_test, (sigmoid(X_test_b @ theta_ada) >= 0.5).astype(int))
print(f"Adagrad — Final loss: {losses_ada[-1]:.4f}  |  Test accuracy: {acc_ada*100:.2f}%")

### 2e. RMSProp

Fix Adagrad's dying learning rate: instead of accumulating ALL past gradients, use an **exponential moving average**.

$$G \leftarrow \rho G + (1-\rho)\nabla^2 \mathcal{L} \qquad \theta \leftarrow \theta - \frac{\alpha}{\sqrt{G + \varepsilon}} \nabla \mathcal{L}$$

With $\rho = 0.9$: old gradients are exponentially down-weighted and eventually forgotten → $G$ stays bounded.

In [ ]:
# ── Cell 14: RMSProp optimizer ────────────────────────────────────────────────
def rmsprop_optimizer(X, y, lr=0.01, rho=0.9, eps=1e-8, n_epochs=200):
    """
    RMSProp — Root Mean Square Propagation (Hinton, 2012 — Coursera lecture).

    Key fix over Adagrad: exponential moving average of squared gradients.
      G  ←  ρ·G + (1-ρ)·∇²          (bounded: does NOT grow to infinity)
      θ  ←  θ - lr / sqrt(G + ε) · ∇

    ρ = 0.9 gives a window of roughly 1/(1-ρ) = 10 past gradient magnitudes.
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    G      = np.zeros(p)   # exponential moving average of squared gradients
    losses = []

    for epoch in range(n_epochs):
        grad = logistic_gradient(X, y, theta)

        # Exponential moving average — old history fades out (unlike Adagrad)
        G    = rho * G + (1 - rho) * grad ** 2

        # Adaptive step: parameters with large recent gradients get small steps
        theta -= (lr / (np.sqrt(G) + eps)) * grad

        losses.append(logistic_loss(X, y, theta))

    return theta, losses

theta_rms, losses_rms = rmsprop_optimizer(X_train_b, y_train, lr=0.01, n_epochs=200)
acc_rms = accuracy_score(y_test, (sigmoid(X_test_b @ theta_rms) >= 0.5).astype(int))
print(f"RMSProp — Final loss: {losses_rms[-1]:.4f}  |  Test accuracy: {acc_rms*100:.2f}%")

### 2f. Adam (Adaptive Moment Estimation)

Adam unifies Momentum (1st moment) and RMSProp (2nd moment):

$$m \leftarrow \beta_1 m + (1-\beta_1)\nabla \mathcal{L} \quad \text{(momentum)}$$
$$v \leftarrow \beta_2 v + (1-\beta_2)(\nabla \mathcal{L})^2 \quad \text{(RMSProp)}$$

**Bias correction** — at $t=1$ both $m$ and $v$ are initialised at 0, so they underestimate the true moments. We correct:
$$\hat{m} = \frac{m}{1-\beta_1^t}, \quad \hat{v} = \frac{v}{1-\beta_2^t}$$

$$\theta \leftarrow \theta - \frac{\alpha \hat{m}}{\sqrt{\hat{v}} + \varepsilon}$$

Defaults: $\beta_1=0.9$, $\beta_2=0.999$, $\alpha=0.001$, $\varepsilon=10^{-8}$.

In [ ]:
# ── Cell 16: Adam optimizer ───────────────────────────────────────────────────
def adam_optimizer(X, y, lr=0.001, n_epochs=200, beta1=0.9, beta2=0.999, eps=1e-8):
    """
    Adam — Adaptive Moment Estimation (Kingma & Ba, 2015).

    Combines:
      • Momentum  (first moment  m): smooths gradient direction
      • RMSProp   (second moment v): adapts step size per parameter

    Bias correction divides by (1 - β^t) to account for zero initialisation
    of m and v, which would otherwise underestimate true moments at early t.

    Parameters
    ----------
    beta1 : exponential decay for first moment  (default 0.9)
    beta2 : exponential decay for second moment (default 0.999)
    eps   : small constant to avoid division by zero (default 1e-8)
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    m      = np.zeros(p)   # first  moment (momentum)
    v      = np.zeros(p)   # second moment (RMSProp-style)
    t      = 0             # step counter for bias correction
    losses = []

    for epoch in range(n_epochs):
        t   += 1

        # ── Compute full-batch gradient ────────────────────────────────────────
        z     = X @ theta
        y_hat = 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))
        grad  = (1.0 / n) * (X.T @ (y_hat - y))

        # ── Update biased first and second moment estimates ────────────────────
        m = beta1 * m + (1 - beta1) * grad        # exponential MA of gradient
        v = beta2 * v + (1 - beta2) * grad ** 2   # exponential MA of grad²

        # ── Bias correction — critical in first few iterations ─────────────────
        m_hat = m / (1 - beta1 ** t)              # corrected first  moment
        v_hat = v / (1 - beta2 ** t)              # corrected second moment

        # ── Parameter update ───────────────────────────────────────────────────
        theta -= lr * m_hat / (np.sqrt(v_hat) + eps)

        # Track loss
        loss = -np.mean(
            y * np.log(y_hat + 1e-10) + (1 - y) * np.log(1 - y_hat + 1e-10)
        )
        losses.append(loss)

    return theta, losses

theta_adam, losses_adam = adam_optimizer(X_train_b, y_train, lr=0.001, n_epochs=200)
acc_adam = accuracy_score(y_test, (sigmoid(X_test_b @ theta_adam) >= 0.5).astype(int))
print(f"Adam  — Final loss: {losses_adam[-1]:.4f}  |  Test accuracy: {acc_adam*100:.2f}%")

## Section 3 — Convergence Comparison

Plot all five optimizers' loss curves on the same axes.  
Annotations highlight which converges fastest and which is smoothest.

In [ ]:
# ── Cell 18: Convergence comparison plot ──────────────────────────────────────
all_losses = {
    "SGD":      losses_sgd,
    "Momentum": losses_mom,
    "Adagrad":  losses_ada,
    "RMSProp":  losses_rms,
    "Adam":     losses_adam,
}

fig, ax = plt.subplots(figsize=(11, 5))

for name, losses in all_losses.items():
    ax.plot(losses, label=name, color=OPT_COLORS[name], linewidth=2)

# Annotate final loss values on the right
for name, losses in all_losses.items():
    ax.annotate(
        f"{losses[-1]:.3f}",
        xy=(len(losses) - 1, losses[-1]),
        xytext=(5, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=OPT_COLORS[name],
    )

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
ax.set_title("Convergence Comparison — All Five Optimizers\n"
             "(Logistic Regression on House Price Classification)", fontsize=13)
ax.legend(loc="upper right", fontsize=10)
ax.set_ylim(0.25, 0.75)

# Annotate observations
ax.text(10,  0.69, "SGD: slow & oscillates",    color=OPT_COLORS["SGD"],      fontsize=9)
ax.text(10,  0.66, "Momentum: smooth overshoot", color=OPT_COLORS["Momentum"], fontsize=9)
ax.text(10,  0.63, "Adam: fast & stable ✓",      color=OPT_COLORS["Adam"],     fontsize=9)

plt.tight_layout()
plt.show()
print("\nFinal losses:", {k: f"{v[-1]:.4f}" for k, v in all_losses.items()})

## Section 4 — Learning Rate Sensitivity: Adam vs SGD

A critical practical difference: SGD requires precise learning-rate tuning; Adam is robust over several orders of magnitude.

In [ ]:
# ── Cell 20: Learning rate sensitivity sweep ──────────────────────────────────
lr_values = [0.0001, 0.001, 0.01, 0.1]   # four orders of magnitude

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for lr in lr_values:
    # SGD run
    _, l_sgd  = sgd_optimizer(X_train_b,  y_train, lr=lr,  n_epochs=200)
    # Adam run
    _, l_adam = adam_optimizer(X_train_b, y_train, lr=lr,  n_epochs=200)

    axes[0].plot(l_sgd,  label=f"lr={lr}")
    axes[1].plot(l_adam, label=f"lr={lr}")

for ax, title in zip(axes, ["SGD (sensitive to lr)", "Adam (robust to lr)"]):
    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel("Loss", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.set_ylim(0.2, 1.0)

plt.suptitle("Learning Rate Sensitivity: SGD vs Adam", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: SGD diverges (loss > 0.9) at lr=0.1,")
print("             Adam converges well across ALL four learning rates.")

## Section 5 — 2-D Loss Landscape with Optimization Trajectories

We project the high-dimensional parameter space onto **two principal components** to visualise the loss surface and compare how SGD and Adam traverse it.

In [ ]:
# ── Cell 22: 2-D loss landscape + trajectory visualisation ───────────────────
# ── Step 1: Collect parameter trajectories ────────────────────────────────────
def sgd_with_trajectory(X, y, lr=0.1, n_epochs=150):
    """SGD that also stores parameter vector after each epoch."""
    n, p   = X.shape
    theta  = np.zeros(p)
    path   = [theta.copy()]
    for _ in range(n_epochs):
        theta -= lr * logistic_gradient(X, y, theta)
        path.append(theta.copy())
    return np.array(path)          # shape (n_epochs+1, p)

def adam_with_trajectory(X, y, lr=0.001, n_epochs=150,
                         beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam that also stores parameter vector after each epoch."""
    n, p   = X.shape
    theta  = np.zeros(p)
    m, v   = np.zeros(p), np.zeros(p)
    path   = [theta.copy()]
    for t in range(1, n_epochs + 1):
        grad  = logistic_gradient(X, y, theta)
        m     = beta1 * m + (1 - beta1) * grad
        v     = beta2 * v + (1 - beta2) * grad ** 2
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
        path.append(theta.copy())
    return np.array(path)

sgd_path  = sgd_with_trajectory(X_train_b,  y_train, lr=0.1,   n_epochs=150)
adam_path = adam_with_trajectory(X_train_b, y_train, lr=0.001, n_epochs=150)

# ── Step 2: Project onto 2 PCA directions ─────────────────────────────────────
# Use SVD-based PCA on the combined trajectories
all_paths = np.vstack([sgd_path, adam_path])   # (2*(n+1), p)
mean_path = all_paths.mean(axis=0)
centered  = all_paths - mean_path
U, S, Vt  = np.linalg.svd(centered, full_matrices=False)
pc1, pc2  = Vt[0], Vt[1]                       # top-2 principal directions

def project(path):
    """Project (n_steps, p) trajectory onto (PC1, PC2)."""
    delta = path - mean_path
    return delta @ pc1, delta @ pc2

# ── Step 3: Build loss surface on the 2-D grid ────────────────────────────────
grid_size = 40
# Range covers both trajectories
xs = np.linspace(-3.5, 3.5, grid_size)
ys = np.linspace(-3.5, 3.5, grid_size)
Z  = np.zeros((grid_size, grid_size))

for i, xi in enumerate(xs):
    for j, yj in enumerate(ys):
        t_ij  = mean_path + xi * pc1 + yj * pc2
        Z[j, i] = logistic_loss(X_train_b, y_train, t_ij)

# ── Step 4: Plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
contour = ax.contourf(xs, ys, Z, levels=30, cmap="RdYlGn_r", alpha=0.8)
plt.colorbar(contour, ax=ax, label="Loss")

# Projected trajectories
sx, sy = project(sgd_path)
ax_x, ay = project(adam_path)

ax.plot(sx, sy, color=OPT_COLORS["SGD"],  linewidth=2, label="SGD",  zorder=5)
ax.plot(ax_x, ay, color=OPT_COLORS["Adam"], linewidth=2, label="Adam", zorder=5)

# Mark start and end
ax.scatter([sx[0]],  [sy[0]],  s=80, color="black",               zorder=6)
ax.scatter([sx[-1]], [sy[-1]], s=80, color=OPT_COLORS["SGD"],     zorder=6, marker="*")
ax.scatter([ax_x[-1]], [ay[-1]], s=80, color=OPT_COLORS["Adam"], zorder=6, marker="*")

ax.set_xlabel("PC-1 direction", fontsize=11)
ax.set_ylabel("PC-2 direction", fontsize=11)
ax.set_title("2-D Loss Landscape — SGD vs Adam Trajectories\n"
             "(Projected onto top-2 PCA directions)", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Section 6 — Bias Correction Effect in Adam

Without bias correction, $m$ and $v$ start at 0 (far from their true expected values). The first few iterations are **under-estimated** — producing overly cautious steps. Bias correction divides by $(1-\beta^t)$ which is tiny at $t=1$, thus **inflating** the estimates back to their true scale.

In [ ]:
# ── Cell 24: Bias correction effect ───────────────────────────────────────────
def adam_no_bias_correction(X, y, lr=0.001, n_epochs=50,
                             beta1=0.9, beta2=0.999, eps=1e-8):
    """
    Adam WITHOUT bias correction — m_hat = m, v_hat = v.
    Only used here to illustrate the effect in early iterations.
    """
    n, p   = X.shape
    theta  = np.zeros(p)
    m, v   = np.zeros(p), np.zeros(p)
    losses = []
    for epoch in range(n_epochs):
        grad  = logistic_gradient(X, y, theta)
        m     = beta1 * m + (1 - beta1) * grad
        v     = beta2 * v + (1 - beta2) * grad ** 2
        # NO bias correction here
        theta -= lr * m / (np.sqrt(v) + eps)
        losses.append(logistic_loss(X, y, theta))
    return losses

# Run both variants for the first 50 epochs
_, losses_adam_bc    = adam_optimizer(X_train_b, y_train, lr=0.001, n_epochs=50)
losses_adam_no_bc    = adam_no_bias_correction(X_train_b, y_train, lr=0.001, n_epochs=50)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(losses_adam_bc,    label="Adam (with bias correction)",    color=OPT_COLORS["Adam"],     linewidth=2.5)
ax.plot(losses_adam_no_bc, label="Adam (WITHOUT bias correction)", color=OPT_COLORS["Momentum"], linewidth=2.5, linestyle="--")

ax.axvspan(0, 10, alpha=0.1, color="red", label="Early iterations (affected most)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Bias Correction Effect in Adam — First 50 Epochs")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Epoch 1  | With BC:", f"{losses_adam_bc[0]:.4f}",
      " | Without BC:", f"{losses_adam_no_bc[0]:.4f}")
print("Epoch 10 | With BC:", f"{losses_adam_bc[9]:.4f}",
      " | Without BC:", f"{losses_adam_no_bc[9]:.4f}")

## Section 7 — Hyperparameter Sensitivity: Adam β₁ and β₂ Sweeps

Default Adam values ($\beta_1=0.9$, $\beta_2=0.999$) are robust for most tasks. Let's verify empirically.

In [ ]:
# ── Cell 26: Beta hyperparameter sensitivity sweep ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# ── β₁ sweep (momentum coefficient) — keep β₂ fixed at 0.999 ─────────────────
beta1_values = [0.7, 0.8, 0.9, 0.99]
colors_b1    = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]

for b1, col in zip(beta1_values, colors_b1):
    _, losses_b1 = adam_optimizer(
        X_train_b, y_train, lr=0.001, n_epochs=200,
        beta1=b1, beta2=0.999
    )
    axes[0].plot(losses_b1, label=f"β₁={b1}", color=col, linewidth=2)

axes[0].set_title("Adam: β₁ Sweep (β₂ fixed at 0.999)", fontsize=12)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=10)
axes[0].set_ylim(0.3, 0.75)

# ── β₂ sweep (RMSProp coefficient) — keep β₁ fixed at 0.9 ────────────────────
beta2_values = [0.9, 0.99, 0.999]
colors_b2    = ["#d62728", "#ff7f0e", "#1f77b4"]

for b2, col in zip(beta2_values, colors_b2):
    _, losses_b2 = adam_optimizer(
        X_train_b, y_train, lr=0.001, n_epochs=200,
        beta1=0.9, beta2=b2
    )
    axes[1].plot(losses_b2, label=f"β₂={b2}", color=col, linewidth=2)

axes[1].set_title("Adam: β₂ Sweep (β₁ fixed at 0.9)", fontsize=12)
axes[1].set_xlabel("Epoch")
axes[1].legend(fontsize=10)

plt.suptitle("Adam Hyperparameter Sensitivity", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: β₁=0.99 (too much momentum) converges slowly.")
print("             β₂=0.9  (too little history) adds noise.")
print("             Defaults (β₁=0.9, β₂=0.999) are robust for most tasks.")

## Section 8 — Convergence Speed Table

How many epochs does each optimizer need to reach a given loss level?

In [ ]:
# ── Cell 28: Convergence speed table ──────────────────────────────────────────
# Re-run all optimizers for 300 epochs to allow all to reach targets
results_300 = {}
results_300["SGD"]      = sgd_optimizer(X_train_b,      y_train, lr=0.1,   n_epochs=300)[1]
results_300["Momentum"] = momentum_optimizer(X_train_b, y_train, lr=0.1,   n_epochs=300)[1]
results_300["Adagrad"]  = adagrad_optimizer(X_train_b,  y_train, lr=0.5,   n_epochs=300)[1]
results_300["RMSProp"]  = rmsprop_optimizer(X_train_b,  y_train, lr=0.01,  n_epochs=300)[1]
results_300["Adam"]     = adam_optimizer(X_train_b,     y_train, lr=0.001, n_epochs=300)[1]

def epochs_to_reach(losses, threshold):
    """Return the first epoch index where loss drops below threshold; NaN if never."""
    for i, loss in enumerate(losses):
        if loss <= threshold:
            return i + 1     # 1-indexed epoch
    return float("nan")      # never reached

# Build comparison table
targets    = [0.55, 0.50, 0.45, 0.40]
table_rows = []

for opt_name, losses in results_300.items():
    row = {"Optimizer": opt_name}
    for tgt in targets:
        row[f"Loss≤{tgt}"] = epochs_to_reach(losses, tgt)
    row["Final Loss (ep 300)"] = f"{losses[-1]:.4f}"
    table_rows.append(row)

speed_df = pd.DataFrame(table_rows).set_index("Optimizer")

print("\nEpochs required to reach each loss threshold (NaN = not reached in 300 epochs)")
print("=" * 70)
print(speed_df.to_string())
print()
print("Adam typically reaches each threshold in the fewest epochs.")

## Self-Check — Three Conceptual Questions

Work through these before looking at the answers.

---

**Q1.** Adam uses bias correction: $\hat{m} = m / (1 - \beta_1^t)$. Why is this needed in the early iterations?

<details>
<summary>Show Answer</summary>

Both $m$ and $v$ are initialised to **zero** before training starts. At iteration $t=1$, the update is:
$$m_1 = \beta_1 \cdot 0 + (1-\beta_1)\nabla = (1-\beta_1)\nabla$$
With $\beta_1=0.9$ this gives $m_1 = 0.1\nabla$ — only **10% of the true gradient**. Without correction, the first steps are 10× too small. Dividing by $(1-\beta_1^1) = 0.1$ recovers the full gradient magnitude. As $t$ grows, $\beta_1^t \to 0$ so the correction factor $\to 1$ and bias correction becomes irrelevant.

</details>

---

**Q2.** Adagrad's learning rate decays to near-zero and training stops. How does RMSProp fix this?

<details>
<summary>Show Answer</summary>

Adagrad accumulates **all** squared gradients from the beginning: $G += \nabla^2$. Since $G$ only grows, the effective learning rate $\alpha / \sqrt{G}$ shrinks monotonically and eventually approaches zero — training stalls.  
RMSProp replaces the cumulative sum with an **exponential moving average**: $G \leftarrow \rho G + (1-\rho)\nabla^2$. Old gradients are down-weighted geometrically (factor $\rho^k$ after $k$ steps), so $G$ stays bounded around the magnitude of *recent* gradients. The learning rate remains useful throughout training.

</details>

---

**Q3.** SGD with momentum $\beta=0.9$ overshoots the minimum and oscillates. What value of $\beta$ would you try to fix this?

<details>
<summary>Show Answer</summary>

Try **reducing $\beta$** — e.g. $\beta = 0.7$ or $\beta = 0.5$. A smaller momentum coefficient means the velocity decays faster and accumulates less kinetic energy, so the optimizer slows down near the minimum instead of overshooting. Alternatively, you can keep $\beta=0.9$ but **reduce the learning rate** — lowering $\alpha$ decreases the size of each step, which also reduces overshooting. In practice, the two hyperparameters are co-tuned: high $\beta$ works best with a lower $\alpha$.

</details>